# Flight Plan Computation

The `flight_plan` module converts a sequence of flight lines (and optional
airports) into a detailed, segment-by-segment flight plan. Each segment
is classified as takeoff, climb, transit, flight_line, descent, or approach
and includes geometry, timing, altitude, and heading information.

We cover:

1. Computing a basic flight plan
2. Understanding segment types
3. Extracting statistics (distance, time, data-collection fraction)
4. Plotting the flight plan on a map
5. Altitude trajectory profiles
6. Flight plans without airports
7. Exporting flight plan data

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd

from hyplan import (
    FlightLine, AVIRIS3,
    DynamicAviation_B200, NASA_ER2,
    Airport, initialize_data,
    compute_flight_plan, plot_flight_plan, plot_altitude_trajectory,
    ureg,
)

## 1. Setup: Flight Lines and Airports

First, we create a few flight lines and select departure/return airports.

In [ ]:
# Initialize airport database
initialize_data(countries=["US"])

# Define flight lines near Santa Barbara
line1 = FlightLine.center_length_azimuth(
    lat=34.4, lon=-119.8,
    length=ureg.Quantity(40, "km"),
    az=90.0,
    altitude_msl=ureg.Quantity(20000, "feet"),
    site_name="Coastal_L01",
)

line2 = FlightLine.center_length_azimuth(
    lat=34.35, lon=-119.8,
    length=ureg.Quantity(40, "km"),
    az=270.0,  # reverse direction
    altitude_msl=ureg.Quantity(20000, "feet"),
    site_name="Coastal_L02",
)

line3 = FlightLine.center_length_azimuth(
    lat=34.3, lon=-119.8,
    length=ureg.Quantity(40, "km"),
    az=90.0,
    altitude_msl=ureg.Quantity(20000, "feet"),
    site_name="Coastal_L03",
)

flight_lines = [line1, line2, line3]

# Select airports
departure = Airport("KSBA")  # Santa Barbara
arrival = Airport("KSBA")    # Return to same airport

print(f"Departure: {departure.name} ({departure.icao_code})")
print(f"  Elevation: {departure.elevation_ft:.0f} ft")
print(f"\n{len(flight_lines)} flight lines:")
for fl in flight_lines:
    print(f"  {fl.site_name}: {fl.length.to(ureg.km):.1f}, alt={fl.altitude_msl}")

## 2. Computing the Flight Plan

`compute_flight_plan` takes an aircraft, a sequence of flight lines,
and optional departure/return airports. It computes:

- Takeoff and climb from the departure airport
- Transit legs between flight lines (using Dubins paths)
- Data-collection segments along each flight line
- Descent and approach to the return airport

The result is a GeoDataFrame with one row per segment.

In [ ]:
aircraft = DynamicAviation_B200()

plan = compute_flight_plan(
    aircraft=aircraft,
    flight_sequence=flight_lines,
    takeoff_airport=departure,
    return_airport=arrival,
)

print(f"Flight plan: {len(plan)} segments")
print(f"Columns: {list(plan.columns)}")

## 3. Understanding Segment Types

Each segment in the flight plan has a `segment_type`:

| Type | Description |
|------|-------------|
| `takeoff` | Initial climb from the departure airport |
| `climb` | Subsequent ascending segments |
| `transit` | Level flight between waypoints |
| `flight_line` | Data-collection segment along a flight line |
| `descent` | Descending segments |
| `approach` | Final descent into the return airport |

In [ ]:
# Display the flight plan table
display_cols = ["segment_type", "segment_name", "distance", "time_to_segment",
                "start_altitude", "end_altitude"]
print(plan[display_cols].to_string(index=False, float_format="%.1f"))

## 4. Flight Plan Statistics

Extract key metrics from the flight plan for mission planning.

In [ ]:
total_distance = plan["distance"].sum()
total_time = plan["time_to_segment"].sum()

# Data-collection segments only
data_segs = plan[plan["segment_type"] == "flight_line"]
data_distance = data_segs["distance"].sum()
data_time = data_segs["time_to_segment"].sum()

# Transit segments
transit_segs = plan[plan["segment_type"] == "transit"]
transit_time = transit_segs["time_to_segment"].sum()

print("Mission Statistics")
print("=" * 40)
print(f"Total distance:       {total_distance:>8.1f} nmi")
print(f"Total flight time:    {total_time:>8.1f} min ({total_time/60:.1f} hr)")
print(f"Data collection:      {data_distance:>8.1f} nmi ({data_time:.1f} min)")
print(f"Transit time:         {transit_time:>8.1f} min")
print(f"Collection fraction:  {data_time/total_time*100:>8.1f}%")
print(f"\nSegment breakdown:")
print(plan.groupby("segment_type")["time_to_segment"].sum().to_string())

## 5. Map Visualization

`plot_flight_plan` shows the entire mission on a 2D map, color-coded
by segment type.

In [ ]:
plot_flight_plan(plan, departure, arrival, flight_lines)

### Interactive Map

Visualize the flight plan segments on an interactive Folium map.

In [ ]:
import folium

# Color-code segments by type
seg_colors = {
    "takeoff": "red", "climb": "orange", "transit": "gray",
    "flight_line": "blue", "descent": "purple", "approach": "darkred",
}

m = folium.Map(
    location=[plan.iloc[0]["start_lat"], plan.iloc[0]["start_lon"]],
    zoom_start=10, tiles="CartoDB positron",
)

# Add each segment as a colored polyline
for _, row in plan.iterrows():
    if row["geometry"] is None or row["geometry"].is_empty:
        continue
    coords = [(lat, lon) for lon, lat in row["geometry"].coords]
    color = seg_colors.get(row["segment_type"], "black")
    popup = (f"<b>{row['segment_name']}</b><br>"
             f"Type: {row['segment_type']}<br>"
             f"Alt: {row['start_altitude']:.0f}→{row['end_altitude']:.0f} ft<br>"
             f"Time: {row['time_to_segment']:.1f} min")
    folium.PolyLine(
        coords, color=color, weight=3, opacity=0.8,
        popup=folium.Popup(popup, max_width=250),
        tooltip=f"{row['segment_type']}: {row['segment_name']}",
    ).add_to(m)

# Airport markers
for apt, label, color in [(departure, "Departure", "green"), (arrival, "Return", "red")]:
    folium.Marker(
        [apt.latitude, apt.longitude],
        popup=f"{apt.name} ({apt.icao_code})",
        icon=folium.Icon(color=color, icon="plane", prefix="fa"),
    ).add_to(m)

m

## 6. Altitude Trajectory

`plot_altitude_trajectory` shows altitude versus time, optionally
with terrain elevation underneath. When an `Aircraft` is provided,
climb segments use realistic curved profiles (rate of climb decreases
with altitude).

In [ ]:
# With aircraft (curved climb profile) — terrain disabled for speed
plot_altitude_trajectory(plan, aircraft=aircraft, show_terrain=False)

In [ ]:
# Without aircraft (straight-line altitude segments)
plot_altitude_trajectory(plan, aircraft=None, show_terrain=False)

## 7. Flight Plan Without Airports

You can also compute a flight plan with no departure or return airport.
This is useful for analyzing just the data-collection portion
and transit between flight lines.

In [ ]:
plan_no_airports = compute_flight_plan(
    aircraft=aircraft,
    flight_sequence=flight_lines,
    # No takeoff_airport or return_airport
)

print(f"Segments without airports: {len(plan_no_airports)}")
print(plan_no_airports[["segment_type", "segment_name", "time_to_segment"]].to_string(index=False, float_format="%.1f"))

## 8. Different Aircraft

The same flight lines produce different plans depending on the aircraft's
climb rate, cruise speed, and service ceiling.

In [ ]:
# Compare B200 vs ER-2 (high-altitude aircraft)
high_lines = [
    FlightLine.center_length_azimuth(
        lat=34.4, lon=-119.8,
        length=ureg.Quantity(50, "km"), az=90.0,
        altitude_msl=ureg.Quantity(65000, "feet"),
        site_name="HighAlt_L01",
    ),
    FlightLine.center_length_azimuth(
        lat=34.35, lon=-119.8,
        length=ureg.Quantity(50, "km"), az=270.0,
        altitude_msl=ureg.Quantity(65000, "feet"),
        site_name="HighAlt_L02",
    ),
]

er2 = NASA_ER2()
er2_plan = compute_flight_plan(
    aircraft=er2,
    flight_sequence=high_lines,
    takeoff_airport=Airport("KPMD"),  # Palmdale
    return_airport=Airport("KPMD"),
)

print(f"ER-2 flight plan: {len(er2_plan)} segments")
print(f"Total time: {er2_plan['time_to_segment'].sum():.1f} min")
print()
print(er2_plan[["segment_type", "segment_name", "start_altitude", "end_altitude", "time_to_segment"]].to_string(
    index=False, float_format="%.1f"
))

In [ ]:
plot_altitude_trajectory(er2_plan, aircraft=er2, show_terrain=False)

## 9. Exporting Flight Plan Data

The flight plan GeoDataFrame can be exported to GeoJSON, shapefiles,
or CSV for use in other tools.

In [ ]:
# Export to GeoJSON
# plan.to_file("flight_plan.geojson", driver="GeoJSON")

# Export to CSV (without geometry)
csv_cols = ["segment_type", "segment_name", "start_lat", "start_lon",
            "end_lat", "end_lon", "start_altitude", "end_altitude",
            "distance", "time_to_segment"]
# plan[csv_cols].to_csv("flight_plan.csv", index=False)

print("Export-ready columns:")
print(plan[csv_cols].to_string(index=False, float_format="%.2f"))

## Summary

| Function | Purpose |
|----------|---------|
| `compute_flight_plan(aircraft, flight_sequence, takeoff_airport, return_airport)` | Full segment-by-segment flight plan |
| `plot_flight_plan(gdf, departure, arrival, flight_lines)` | 2D map of the flight plan |
| `plot_altitude_trajectory(gdf, aircraft, dem_file, show_terrain)` | Altitude vs. time profile |

**Key concepts:**
- The flight plan includes takeoff, climb, transit, data-collection, descent, and approach segments
- Transit legs use Dubins minimum-radius turning paths between flight line endpoints
- Climb profiles are realistic when an `Aircraft` object is provided
- The `start_offset` and `end_offset` parameters add buffer distance before/after each flight line
- The resulting GeoDataFrame is exportable to standard geospatial formats